# Train/validation/test splits and scaler fitting (v3)

Reads the CRSP IT-sector merged file `files/sp500_it_complete_1999_2024.csv.gz` (prepared per `REPRODUCIBILITY.md` §"Data provenance"), engineers the eight rolling features, drops rolling-feature warmup rows, splits chronologically into 1999–2018 (train), 2019–2021 (validation), and 2022–2024 (test), fits the z-score scaler on the training window only (the Sprint 3 leakage fix), applies it to all three splits, and writes `results/scaler_params.csv` and `files/{train,val,test}_data_v3.csv.gz`.

Requires `FRED_API_KEY` set in the environment for the VIX merge.


# Imports


In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
from fredapi import Fred

warnings.filterwarnings('ignore')


# Load CRSP and clean


In [ ]:
df = pd.read_csv('files/sp500_it_complete_1999_2024.csv.gz', compression='gzip',
                 parse_dates=['date'], dtype={'TICKER': str, 'RET': str}, low_memory=False)

df['RET'] = pd.to_numeric(df['RET'], errors='coerce')
df['PRC'] = df['PRC'].abs()
df['BIDLO'] = df['BIDLO'].abs()
df['ASKHI'] = df['ASKHI'].abs()
df = df.dropna(subset=['PRC'])

df['CFACPR'] = pd.to_numeric(df['CFACPR'], errors='coerce').fillna(1)
df['CFACSHR'] = pd.to_numeric(df['CFACSHR'], errors='coerce').fillna(1)
df['adj_prc'] = df['PRC'] / df['CFACPR']
df['mkt_cap'] = df['PRC'] * df['SHROUT'] * 1000


# Merge VIX (FRED)


In [ ]:
fred = Fred(api_key=os.environ['FRED_API_KEY'])
vix = fred.get_series('VIXCLS', observation_start='1999-01-01').reset_index()
vix.columns = ['date', 'vix_index']
vix['vix_index'] = pd.to_numeric(vix['vix_index'], errors='coerce')
df = pd.merge(df, vix, on='date', how='left')
df['vix_index'] = df['vix_index'].ffill().bfill()


# Resolve ticker changes (carry the most recent TICKER per PERMNO)


In [ ]:
latest_ticker = df.sort_values('date').groupby('PERMNO')['TICKER'].last()
df['TICKER'] = df['PERMNO'].map(latest_ticker.to_dict())
df = df.sort_values(['PERMNO', 'date']).reset_index(drop=True)


# Feature engineering (RSI, MACD, Bollinger %B, ATR, rolling vol, SMA-20/50, log volume)


In [ ]:
def compute_rsi(prices, period=14):
    delta = prices.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.rolling(period, min_periods=period).mean()
    avg_loss = loss.rolling(period, min_periods=period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

df['RSI'] = df.groupby('PERMNO')['adj_prc'].transform(lambda x: compute_rsi(x))
df['MACD'] = df.groupby('PERMNO')['adj_prc'].transform(
    lambda x: x.ewm(span=12).mean() - x.ewm(span=26).mean())

bb_mid = df.groupby('PERMNO')['adj_prc'].transform(lambda x: x.rolling(20, min_periods=20).mean())
bb_std = df.groupby('PERMNO')['adj_prc'].transform(lambda x: x.rolling(20, min_periods=20).std())
df['BB_PCT'] = (df['adj_prc'] - (bb_mid - 2*bb_std)) / (4*bb_std)

def compute_atr(group):
    tr = pd.concat([
        group['ASKHI'] - group['BIDLO'],
        (group['ASKHI'] - group['adj_prc'].shift(1)).abs(),
        (group['BIDLO'] - group['adj_prc'].shift(1)).abs()
    ], axis=1).max(axis=1)
    return tr.rolling(14, min_periods=14).mean()

df['ATR'] = df.groupby('PERMNO').apply(compute_atr).reset_index(level=0, drop=True)
df['rolling_vol_60'] = df.groupby('PERMNO')['RET'].transform(lambda x: x.rolling(60, min_periods=60).std())
df['SMA_20'] = df.groupby('PERMNO')['adj_prc'].transform(lambda x: x.rolling(20, min_periods=20).mean())
df['SMA_50'] = df.groupby('PERMNO')['adj_prc'].transform(lambda x: x.rolling(50, min_periods=50).mean())
df['log_vol'] = np.log1p(df['VOL'])


# Drop rolling-feature warmup rows


In [ ]:
df = df.dropna(subset=['RSI', 'MACD', 'BB_PCT', 'ATR', 'rolling_vol_60', 'SMA_50', 'vix_index'])


# Chronological train / validation / test split (1999–2018 / 2019–2021 / 2022–2024)


In [ ]:
train_mask = df['date'] < '2019-01-01'
val_mask = (df['date'] >= '2019-01-01') & (df['date'] < '2022-01-01')
test_mask = df['date'] >= '2022-01-01'


# Preserve raw returns before normalization (env uses raw_RET for reward)


In [ ]:
df['raw_RET'] = df['RET'].copy()


# Fit scaler on training window and save scaler_params.csv (Sprint 3 leakage fix)


In [ ]:
features_to_scale = ['adj_prc', 'log_vol', 'RET', 'mkt_cap', 'RSI', 'MACD',
                     'BB_PCT', 'ATR', 'rolling_vol_60', 'SMA_20', 'SMA_50',
                     'BIDLO', 'ASKHI', 'OPENPRC', 'vix_index']

train_mean = df.loc[train_mask, features_to_scale].mean()
train_std = df.loc[train_mask, features_to_scale].std().replace(0, 1)

# Save scaler params so training/evaluation scripts can denormalize
scaler_params = pd.DataFrame({'mean': train_mean, 'std': train_std})
scaler_params.to_csv('results/scaler_params.csv')
print(f"  Saved scaler_params.csv with {len(scaler_params)} features")
print(f"  RET mean={train_mean['RET']:.6f}, std={train_std['RET']:.6f}")


# Apply scaler and save v3 splits


In [ ]:
# Normalize (only the feature columns, NOT raw_RET)
df[features_to_scale] = (df[features_to_scale] - train_mean) / train_std

df[train_mask].to_csv('files/train_data_v3.csv.gz', index=False, compression='gzip')
df[val_mask].to_csv('files/val_data_v3.csv.gz', index=False, compression='gzip')
df[test_mask].to_csv('files/test_data_v3.csv.gz', index=False, compression='gzip')

# Print summary
for name, mask in [('Train', train_mask), ('Val', val_mask), ('Test', test_mask)]:
    subset = df[mask]
    print(f"  {name}: {len(subset)} rows, {subset['PERMNO'].nunique()} stocks, "
          f"{subset['date'].min().date()} to {subset['date'].max().date()}")

print(f"\nDONE! {df['PERMNO'].nunique()} stocks, {len(df)} rows total.")
print(f"Key files: scaler_params.csv, train/val/test_data_v3.csv.gz")
print(f"New column 'raw_RET' contains un-normalized returns for reward computation.")
